![Teste vocacional com Machine Learning](capa.png)


# Projeto - Teste Vocacional com Machine Learning
## Etapa 01 - Coleta de Dados

## O Problema de Negócio

Escolher um curso superior é uma das decisões mais importantes na vida de um jovem, e muitas vezes ela é tomada com pouca informação: por pressão familiar, pela fama da profissão ou por influência dos amigos. O resultado aparece nos números: uma parcela grande dos estudantes troca de curso ou abandona a graduação nos primeiros anos.

Este projeto propõe um **teste vocacional baseado em dados**, que usa Machine Learning para relacionar o **perfil de interesses** de uma pessoa às **áreas de formação** que mais combinam com ela.

## Objetivo

Responder à pergunta: **"dado o perfil de interesses de uma pessoa, quais áreas de formação combinam mais com ela?"**

O sistema recomendará as **3 áreas mais compatíveis** e explicará quais características do perfil pesaram em cada recomendação.

## Fundamentação: o modelo RIASEC de Holland

O projeto usa o **modelo RIASEC**, criado pelo psicólogo John Holland e amplamente adotado em orientação vocacional. Ele classifica os interesses em seis dimensões:

| Sigla | Dimensão | Perfil |
|---|---|---|
| **R** | Realista | gosta de atividades práticas, ferramentas, máquinas, natureza |
| **I** | Investigativo | gosta de pesquisar, analisar, resolver problemas |
| **A** | Artístico | gosta de criar, se expressar, inovar |
| **S** | Social | gosta de ajudar, ensinar, cuidar de pessoas |
| **E** | Empreendedor | gosta de liderar, persuadir, negociar |
| **C** | Convencional | gosta de organização, dados, rotinas e processos |

## Fontes de Dados

- **Open Psychometrics (RIASEC, 48 itens):** respostas públicas de milhares de pessoas ao teste RIASEC, incluindo o curso/formação de cada uma. É a **base de treino** do modelo.
- **INEP - Censo da Educação Superior e classificação CINE Brasil:** contexto brasileiro (cursos mais procurados, concluintes) e padronização das áreas de formação.
- **MEC - Catálogo de Cursos e Diretrizes Curriculares:** descrição do que cada curso estuda.

## Etapas do Projeto (Pipeline de Ciência de Dados)

1. Coleta de dados *(este notebook)*
2. Limpeza e engenharia de atributos
3. Análise exploratória dos dados (EDA)
4. Modelagem
5. Avaliação do modelo
6. Deploy da aplicação web

## Considerações Éticas

- O sistema **sugere, não determina**: o resultado é um ponto de partida para reflexão, não uma sentença sobre o futuro de ninguém.
- Não são coletados dados pessoais identificáveis, em respeito à **LGPD**.
- **Limitação conhecida:** a base de treino tem respondentes majoritariamente de países de língua inglesa, o que pode introduzir viés. Esse ponto será analisado e documentado ao longo do projeto.

In [80]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto:', python_version())


Versão da Linguagem Python usada neste Projeto: 3.14.4


In [81]:
# Os pacotes deste projeto foram instalados no ambiente virtual (.venv),
# pelo terminal do VS Code, com o ambiente ativado:
#
#   .venv\Scripts\Activate.ps1
#   pip install pandas numpy matplotlib seaborn ipykernel watermark
#
# Para atualizar um pacote:
#   pip install -U nome_pacote
#
# Para instalar uma versão exata de um pacote:
#   pip install nome_pacote==versão_desejada
#
# Depois de instalar ou atualizar um pacote, reinicie o kernel do notebook.


### Importando as bibliotecas necessárias


In [82]:
# Para manipulação dos dados a dupla inseparavel 
import pandas as pd 
import numpy as np

In [83]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Gustavo" --iversions


Author: Gustavo

numpy   : 2.5.3
pandas  : 3.0.6
platform: 1.0.9



#### Começando a analise do dataframe 

In [84]:
# Importando o dataset RIASEC (Open Psychometrics)
df = pd.read_csv('../dados/Dados originais/data.csv', sep='\t')
df.head(10)


C:\Users\Gusta\AppData\Local\Temp\ipykernel_93140\773830837.py:2: DtypeWarning: Columns (93: Unnamed: 93) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dados/Dados originais/data.csv', sep='\t')


,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major,Unnamed: 93
0,3,4,3,1,1,4,1,3,5,5,...,1,1,2,1,1,1,US,2,NaN,NaN
1,1,1,2,4,1,2,2,1,5,5,...,3,4,1,2,3,1,US,1,Nursing,NaN
2,2,1,1,1,1,1,1,1,4,1,...,1,4,2,1,1,1,US,1,NaN,NaN
3,3,1,1,2,2,2,2,2,4,1,...,1,1,2,1,1,1,CN,0,NaN,NaN
4,4,1,1,2,1,1,1,2,5,5,...,3,1,2,1,4,1,PH,0,education,NaN
5,3,5,1,3,1,5,3,4,4,5,...,1,5,1,1,2,1,IN,2,Literature,NaN
6,1,4,1,4,1,4,1,2,4,4,...,1,1,2,1,3,2,US,0,Math,NaN
7,5,1,2,2,2,1,2,1,4,4,...,3,1,2,1,0,1,PH,0,mathematics and science,NaN
8,1,1,1,1,1,1,1,1,2,1,...,1,1,1,1,3,1,US,0,family resource management,NaN
9,4,2,2,2,2,2,3,2,2,2,...,1,1,2,1,6,1,MY,1,Counselling,NaN


### Como podemos observar há valores nulos na coluna Major e uma coluna fantasma sem valores unnamed93

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145828 entries, 0 to 145827
Data columns (total 94 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   R1                     145828 non-null  int64
 1   R2                     145828 non-null  int64
 2   R3                     145828 non-null  int64
 3   R4                     145828 non-null  int64
 4   R5                     145828 non-null  int64
 5   R6                     145828 non-null  int64
 6   R7                     145828 non-null  int64
 7   R8                     145828 non-null  int64
 8   I1                     145828 non-null  int64
 9   I2                     145828 non-null  int64
 10  I3                     145828 non-null  int64
 11  I4                     145828 non-null  int64
 12  I5                     145828 non-null  int64
 13  I6                     145828 non-null  int64
 14  I7                     145828 non-null  int64
 15  I8                     14582

#### descrição dos dados media, min, max....

In [86]:
df.describe()
#descreve o dataet 

,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,age,hand,religion,orientation,race,voted,married,familysize,uniqueNetworkLocation,source
count,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,...,1.458280e+05,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,1.458280e+05,145828.000000,145828.000000
mean,2.572846,2.106859,1.748738,2.293640,1.749945,2.197815,2.017020,1.967215,3.434752,3.335669,...,1.178402e+05,1.136757,5.481375,1.363497,3.287846,1.620827,1.274913,1.255801e+05,1.309275,0.420242
std,1.318347,1.235213,1.120873,1.339156,1.072175,1.273549,1.182839,1.184509,1.333190,1.362869,...,1.533419e+07,0.416572,3.417863,1.041039,1.400790,0.512136,0.566570,1.612271e+07,0.462196,0.651346
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.300000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,...,1.700000e+01,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000,2.000000e+00,1.000000,0.000000
50%,3.000000,2.000000,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,4.000000,4.000000,...,2.100000e+01,1.000000,6.000000,1.000000,4.000000,2.000000,1.000000,3.000000e+00,1.000000,0.000000
75%,3.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,3.000000,5.000000,4.000000,...,3.100000e+01,1.000000,7.000000,1.000000,4.000000,2.000000,1.000000,3.000000e+00,2.000000,1.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,2.147484e+09,3.000000,12.000000,5.000000,5.000000,2.000000,3.000000,2.147484e+09,2.000000,2.000000


In [87]:
df.shape
#Numero de linhas e colunas do data set

(145828, 94)

#### tratando valores auzentes

In [88]:
def calcular_porcentagem_valores_ausentes(data):
 
    # Calcula o total de células do dataset
    total_de_celulas = np.prod(data.shape)
    # Conta o número de valores ausentes por coluna
    quantidade_ausentes_por_coluna = data.isnull().sum()
    # Calcula o total de valores ausentes em todo o conjunto de dados
    total_ausentes = quantidade_ausentes_por_coluna.sum()
    # Calcula a porcentagem de valores ausentes em relação ao total de células
    porcentagem_ausentes = (total_ausentes / total_de_celulas) * 100
    # Exibe a porcentagem de valores ausentes no conjunto de dados
 
    print(f"O conjunto de dados tem {round(porcentagem_ausentes, 2)}% de valores ausentes.")


In [89]:
# Verificando a porcentagem de valores ausentes no dataset
calcular_porcentagem_valores_ausentes(df)


O conjunto de dados tem 1.45% de valores ausentes.


In [90]:
# Versão com return: DEVOLVE a porcentagem em vez de apenas mostrar na tela
def obter_porcentagem_valores_ausentes(data):

    # Calcula o total de células do dataset
    total_de_celulas = np.prod(data.shape)
    # Conta o número de valores ausentes por coluna
    quantidade_ausentes_por_coluna = data.isnull().sum()
    # Calcula o total de valores ausentes em todo o conjunto de dados
    total_ausentes = quantidade_ausentes_por_coluna.sum()
    # Calcula a porcentagem de valores ausentes em relação ao total de células
    porcentagem_ausentes = (total_ausentes / total_de_celulas) * 100
    # Devolve a porcentagem para quem chamou a função
    return porcentagem_ausentes

In [91]:
#Guardando a função da variavel
porcentagem = obter_porcentagem_valores_ausentes(df)

#usando o resultado para tomar decisão
if porcentagem > 5: 
    print(f"atenção: {porcentagem:.2f}% de ausentes, preciisamos tratar!")
else:
    print(f"Apenas {porcentagem:.2f}% de ausentes, dataset em bom estado.")


Apenas 1.45% de ausentes, dataset em bom estado.


In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145828 entries, 0 to 145827
Data columns (total 94 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   R1                 145828 non-null  int64
 1   R2                 145828 non-null  int64
 2   R3                 145828 non-null  int64
 3   R4                 145828 non-null  int64
 4   R5                 145828 non-null  int64
 5   R6                 145828 non-null  int64
 6   R7                 145828 non-null  int64
 7   R8                 145828 non-null  int64
 8   I1                 145828 non-null  int64
 9   I2                 145828 non-null  int64
 10  I3                 145828 non-null  int64
 11  I4                 145828 non-null  int64
 12  I5                 145828 non-null  int64
 13  I6                 145828 non-null  int64
 14  I7                 145828 non-null  int64
 15  I8                 145828 non-null  int64
 16  A1                 145828 non-null  int64
 17  A2

#### Como podemos perceber no df.info() 
 93  Unnamed: 93        1 non-null       str 
 #### A coluna possui um único valor e é preciso analisar

In [95]:
#Filtrando apenas a linha que contem o valor na coluna Unnamed
linha_fantasma = df[df['Unnamed: 93'].notna()]
linha_fantasma.T


,122158
R1,2
R2,1
R3,3
R4,2
R5,1
...,...
rede_unica,1
pais,RU
origem_acesso,0
curso,NaN


#### Um participante russo que cursou economia. Provavelmente ele digitou um TAB antes da palavra quando preencheu o curso. Como o arquivo usa TAB como separador, o "economics" foi empurrado uma coluna para a direita, e o campo major ficou vazio.